In [ ]:
import os
import subprocess
import sys
from pathlib import Path

SRC = Path.cwd().parent / "src"   # the notebook is in notebooks/, minimal.py is in src/

def run(script, *args):
    """Runs a script from src/, showing its output live.
    Interrupting the cell (■ button / Interrupt) kills the process and all its children."""
    cmd = [sys.executable, "-u", script, *args]   # -u: unbuffered output
    flags = subprocess.CREATE_NEW_PROCESS_GROUP if os.name == "nt" else 0
    proc = subprocess.Popen(
        cmd,
        cwd=SRC,
        stdin=subprocess.PIPE,                    # minimal.py expects input(); keep stdin open
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,                 # logging goes to stderr: merge it with stdout
        text=True,
        bufsize=1,
        creationflags=flags,
        start_new_session=(os.name != "nt"),
    )
    try:
        for line in proc.stdout:
            print(line, end="", flush=True)
        proc.wait()
    except KeyboardInterrupt:
        print("\n[Interrupted: stopping the process...]")
    finally:
        if proc.poll() is None:
            if os.name == "nt":
                subprocess.run(["taskkill", "/F", "/T", "/PID", str(proc.pid)],
                               stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
            else:
                import signal
                os.killpg(proc.pid, signal.SIGKILL)
            proc.wait()
        print(f"[Process finished, exit code {proc.returncode}]")

# Práctica: Ocultando el Jitter

### Experimento 1: Comunicación Inter-redes (con compañero).
**Enunciado:** The minimum `artificial_latency` that you must configure in your InterCom instance to communicate with your groupmate without lossing chunks, when he/she is in a different8 local network. Ping can help to estimate the latency. However, notice that you must use InterCom (buffer, in this case) to check how many chunks are lost as a function of the buffer size.

#### Comando ejecutado en la terminal:
```bash 
python buffer.py --destination_IP 10.232.195.14 --buffering_time 0
```

### Experimento 2: Simulación Localhost con tc.

**Enunciado:** Repeat the previous experiment using only ``localhost`` (transmiting the chunk between the loopback network adapter), but adding some jitter with the tool tc. If tc is not avaiable in your OS, you can use any other similar tool, or use the Linux at the laboratory. But remember, you only must play with jitter, not with packet loss or transmission bit-rate.